In [81]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt 
%matplotlib inline

In [82]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [83]:
len(words)

32033

In [84]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [85]:
# dataset

block_size = 3
X, Y =[], []
for w in words[:5]:

    print(w)
    context = [0]*block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '---->', itos[ix])
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ----> e
..e ----> m
.em ----> m
emm ----> a
mma ----> .
olivia
... ----> o
..o ----> l
.ol ----> i
oli ----> v
liv ----> i
ivi ----> a
via ----> .
ava
... ----> a
..a ----> v
.av ----> a
ava ----> .
isabella
... ----> i
..i ----> s
.is ----> a
isa ----> b
sab ----> e
abe ----> l
bel ----> l
ell ----> a
lla ----> .
sophia
... ----> s
..s ----> o
.so ----> p
sop ----> h
oph ----> i
phi ----> a
hia ----> .


In [86]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [87]:
C = torch.randn((27, 2))

In [88]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [89]:
w1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [90]:
# torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1).shape    inefficient way of doing it as its creating a new tensor. 

In [91]:
# torch.cat(torch.unbind(emb, 1), 1).shape

In [92]:
h = torch.tanh(emb.view(-1, 6) @ w1 + b1)    # -1 is used to infer the size of the first dimension based on the total number of elements and the size of the other dimensions.
                                            #Also broadcasting happens here.

In [93]:
h

tensor([[ 0.9620,  0.7405,  0.9974,  ...,  0.3920,  0.2951, -0.9562],
        [ 0.3995,  0.9369,  0.9896,  ..., -0.4658,  0.6993, -0.6364],
        [ 0.9987, -0.6711,  0.9235,  ...,  0.2542, -0.1806, -0.9753],
        ...,
        [ 0.4251, -0.9988,  1.0000,  ...,  0.8202, -0.8284, -0.9058],
        [-0.9556, -1.0000, -0.2103,  ..., -0.9946, -0.8947,  0.9503],
        [-0.9945, -0.9907, -1.0000,  ..., -1.0000, -0.7654,  0.9980]])

In [94]:
h.shape

torch.Size([32, 100])

In [95]:
w2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [96]:
logits = h @ w2 + b2

In [97]:
logits.shape

torch.Size([32, 27])

In [98]:
counts = logits.exp()

In [99]:
prob = counts / counts.sum(1, keepdim=True)

In [100]:
prob.shape

torch.Size([32, 27])

### Creating the forward pass 

In [102]:
X.shape, Y.shape

(torch.Size([32, 3]), torch.Size([32]))

In [103]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2), generator=g)
w1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
w2 = torch.rand((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, w1, w2, b1, b2]

In [104]:
sum(p.nelement() for p in parameters)

3481

In [110]:
for p in parameters:
    p.requires_grad = True

In [ ]:
for _ in range(1000):
    # Forward pass
    emb = C[X]   # [32, 3, 2]
    h = torch.tanh(emb.view(-1,6) @w1 +b1)   # [32, 100]
    logits = h @ w2 + b2        # [32, 27]
    # counts = logits.exp()
    # prob = counts / counts.sum(1, keepdim=True)
    # loss =  -prob[torch.arange(32), Y].log().mean()  It is inefficient to create all these tensors so we use the function so we use the function below to find the loss from the input logits and the target Y.
    loss = F.cross_entropy(logits, Y)
    


    # Backward pass
    for p in parameters:
        p.grad = None       # to prevent accumulation of gradients from previous iterations
    loss.backward()
    # Update
    for p in parameters:
        p.data += -0.1 * p.grad

print(loss.item())

0.26028525829315186
